In [1]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


print("=" * 100)
print("03.04A — REPRODUCIBLE BASELINE TRAINING")
print("=" * 100)

print("\nRequired libraries imported successfully.")

print("PyTorch version :", torch.__version__)
print("NumPy version   :", np.__version__)

03.04A — REPRODUCIBLE BASELINE TRAINING

Required libraries imported successfully.
PyTorch version : 2.13.0+cpu
NumPy version   : 2.5.2


In [2]:
# ============================================================
# CELL 2 — REPRODUCIBILITY CONFIGURATION
# ============================================================

SEED = 42


# ------------------------------------------------------------
# Python random seed
# ------------------------------------------------------------

random.seed(SEED)


# ------------------------------------------------------------
# NumPy random seed
# ------------------------------------------------------------

np.random.seed(SEED)


# ------------------------------------------------------------
# PyTorch random seed
# ------------------------------------------------------------

torch.manual_seed(SEED)


# ------------------------------------------------------------
# CUDA seed configuration
# ------------------------------------------------------------

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Deterministic behavior
# ------------------------------------------------------------

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# Select computation device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("=" * 100)
print("REPRODUCIBILITY CONFIGURATION")
print("=" * 100)

print("Random seed :", SEED)
print("Device      :", device)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

print(
    "Deterministic cuDNN:",
    torch.backends.cudnn.deterministic
)

print(
    "cuDNN benchmark:",
    torch.backends.cudnn.benchmark
)

REPRODUCIBILITY CONFIGURATION
Random seed : 42
Device      : cpu
CUDA available: False
Deterministic cuDNN: True
cuDNN benchmark: False


In [3]:
# ============================================================
# CELL 3 — LOCATE PROJECT ROOT
# ============================================================

current_dir = Path.cwd()

project_root = None


# ------------------------------------------------------------
# Search current directory and all parent directories
# ------------------------------------------------------------

for path in [current_dir] + list(current_dir.parents):

    candidate_pair_dir = (
        path
        / "06_cross_modal_learning"
        / "02_cross_modal_pairing"
        / "pairs"
        / "representation_aware_split"
    )

    if candidate_pair_dir.exists():

        project_root = path
        break


# ------------------------------------------------------------
# Stop if project root cannot be found
# ------------------------------------------------------------

if project_root is None:

    raise FileNotFoundError(
        "Could not locate the ASD Multimodal project root.\n"
        "Expected directory structure:\n"
        "06_cross_modal_learning/"
        "02_cross_modal_pairing/"
        "pairs/"
        "representation_aware_split/"
    )


# ------------------------------------------------------------
# Define pair directory
# ------------------------------------------------------------

pair_dir = (
    project_root
    / "06_cross_modal_learning"
    / "02_cross_modal_pairing"
    / "pairs"
    / "representation_aware_split"
)


print("=" * 100)
print("PROJECT ROOT DISCOVERY")
print("=" * 100)

print("Current notebook directory:")
print(current_dir)

print("\nProject root:")
print(project_root)

print("\nPair directory:")
print(pair_dir)

print("\nPair directory exists:",
      pair_dir.exists())

PROJECT ROOT DISCOVERY
Current notebook directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_training\notebooks

Project root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Pair directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split

Pair directory exists: True


In [4]:
# ============================================================
# CELL 4 — LOAD EXACT PHASE 02.08 PAIR DATA
# ============================================================

train_npz_path = (
    pair_dir
    / "02_08_train_representation_disjoint_pairs.npz"
)

val_npz_path = (
    pair_dir
    / "02_08_validation_representation_disjoint_pairs.npz"
)


# ------------------------------------------------------------
# Verify files
# ------------------------------------------------------------

if not train_npz_path.exists():

    raise FileNotFoundError(
        f"Training pair file not found:\n{train_npz_path}"
    )


if not val_npz_path.exists():

    raise FileNotFoundError(
        f"Validation pair file not found:\n{val_npz_path}"
    )


# ------------------------------------------------------------
# Load training pairs
# ------------------------------------------------------------

train_data = np.load(train_npz_path)

train_source = train_data["source_embeddings"]
train_target = train_data["target_embeddings"]
train_labels = train_data["pair_labels"]


# ------------------------------------------------------------
# Load validation pairs
# ------------------------------------------------------------

val_data = np.load(val_npz_path)

val_source = val_data["source_embeddings"]
val_target = val_data["target_embeddings"]
val_labels = val_data["pair_labels"]


print("=" * 100)
print("PHASE 02.08 DATA LOADED")
print("=" * 100)

print("\nTraining")
print("-" * 100)
print("Source embeddings :", train_source.shape)
print("Target embeddings :", train_target.shape)
print("Labels            :", train_labels.shape)

print("\nValidation")
print("-" * 100)
print("Source embeddings :", val_source.shape)
print("Target embeddings :", val_target.shape)
print("Labels            :", val_labels.shape)

PHASE 02.08 DATA LOADED

Training
----------------------------------------------------------------------------------------------------
Source embeddings : (1776, 32)
Target embeddings : (1776, 32)
Labels            : (1776,)

Validation
----------------------------------------------------------------------------------------------------
Source embeddings : (448, 32)
Target embeddings : (448, 32)
Labels            : (448,)


In [5]:
# ============================================================
# CELL 5 — DATA INTEGRITY AUDIT
# ============================================================

print("=" * 100)
print("03.04A PRE-TRAINING DATA INTEGRITY AUDIT")
print("=" * 100)


# ------------------------------------------------------------
# Shape checks
# ------------------------------------------------------------

assert train_source.shape == (1776, 32)
assert train_target.shape == (1776, 32)
assert train_labels.shape == (1776,)

assert val_source.shape == (448, 32)
assert val_target.shape == (448, 32)
assert val_labels.shape == (448,)


# ------------------------------------------------------------
# Label checks
# ------------------------------------------------------------

assert set(np.unique(train_labels)).issubset({0, 1})
assert set(np.unique(val_labels)).issubset({0, 1})


# ------------------------------------------------------------
# Class balance checks
# ------------------------------------------------------------

train_positive = int(np.sum(train_labels == 1))
train_negative = int(np.sum(train_labels == 0))

val_positive = int(np.sum(val_labels == 1))
val_negative = int(np.sum(val_labels == 0))


assert train_positive == 888
assert train_negative == 888

assert val_positive == 224
assert val_negative == 224


# ------------------------------------------------------------
# Numerical validity checks
# ------------------------------------------------------------

assert np.isfinite(train_source).all()
assert np.isfinite(train_target).all()
assert np.isfinite(train_labels).all()

assert np.isfinite(val_source).all()
assert np.isfinite(val_target).all()
assert np.isfinite(val_labels).all()


print("\nDATASET COUNTS")
print("-" * 100)

print("Training pairs        :", len(train_labels))
print("Training positive     :", train_positive)
print("Training negative     :", train_negative)

print("\nValidation pairs      :", len(val_labels))
print("Validation positive   :", val_positive)
print("Validation negative   :", val_negative)


print("\nVALIDATION RESULTS")
print("-" * 100)

print("✓ Source dimensions verified")
print("✓ Target dimensions verified")
print("✓ 32D representation verified")
print("✓ Binary labels verified")
print("✓ Class counts verified")
print("✓ NaN / infinite values absent")


print("\n" + "=" * 100)
print("✓ PRE-TRAINING DATA INTEGRITY AUDIT PASSED")
print("=" * 100)

03.04A PRE-TRAINING DATA INTEGRITY AUDIT

DATASET COUNTS
----------------------------------------------------------------------------------------------------
Training pairs        : 1776
Training positive     : 888
Training negative     : 888

Validation pairs      : 448
Validation positive   : 224
Validation negative   : 224

VALIDATION RESULTS
----------------------------------------------------------------------------------------------------
✓ Source dimensions verified
✓ Target dimensions verified
✓ 32D representation verified
✓ Binary labels verified
✓ Class counts verified
✓ NaN / infinite values absent

✓ PRE-TRAINING DATA INTEGRITY AUDIT PASSED


In [6]:
# ============================================================
# CELL 6 — BASELINE CROSS-MODAL MODEL ARCHITECTURE
# ============================================================

class BaselineCrossModalModel(nn.Module):
    """
    Baseline cross-modal relationship classifier.

    Inputs:
        Source embedding: (batch_size, 32)
        Target embedding: (batch_size, 32)

    Relationship representation:
        [S, T, |S-T|, S*T] -> 128 dimensions

    Classifier:
        128 -> 128 -> 64 -> 1
    """

    def __init__(self):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, source, target):

        # ----------------------------------------------------
        # Relationship features
        # ----------------------------------------------------

        difference = torch.abs(source - target)

        interaction = source * target

        combined = torch.cat(
            [
                source,
                target,
                difference,
                interaction
            ],
            dim=1
        )

        # ----------------------------------------------------
        # Relationship classification
        # ----------------------------------------------------

        output = self.classifier(combined)

        return output.squeeze(1)


# ------------------------------------------------------------
# Create fresh baseline model
# ------------------------------------------------------------

model = BaselineCrossModalModel().to(device)


# ------------------------------------------------------------
# Architecture verification
# ------------------------------------------------------------

print("=" * 100)
print("BASELINE CROSS-MODAL MODEL")
print("=" * 100)

print(model)

print("\nModel device:", device)

BASELINE CROSS-MODAL MODEL
BaselineCrossModalModel(
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

Model device: cpu


In [7]:
# ============================================================
# CELL 7 — FORWARD-PASS VERIFICATION
# ============================================================

model.eval()

with torch.no_grad():

    test_source = torch.tensor(
        train_source[:32],
        dtype=torch.float32
    ).to(device)

    test_target = torch.tensor(
        train_target[:32],
        dtype=torch.float32
    ).to(device)

    test_logits = model(
        test_source,
        test_target
    )


print("=" * 100)
print("BASELINE FORWARD-PASS VERIFICATION")
print("=" * 100)

print("Source input shape :", tuple(test_source.shape))
print("Target input shape :", tuple(test_target.shape))
print("Output shape       :", tuple(test_logits.shape))

print("\nFirst 5 logits:")
print(test_logits[:5])

print(
    "\nAll logits finite:",
    bool(torch.isfinite(test_logits).all())
)

BASELINE FORWARD-PASS VERIFICATION
Source input shape : (32, 32)
Target input shape : (32, 32)
Output shape       : (32,)

First 5 logits:
tensor([0.0996, 0.1113, 0.1294, 0.1088, 0.0885])

All logits finite: True


In [8]:
# ============================================================
# CELL 8 — BCE WITH LOGITS LOSS VERIFICATION
# ============================================================

criterion = nn.BCEWithLogitsLoss()

test_labels = torch.tensor(
    train_labels[:32],
    dtype=torch.float32
).to(device)


test_loss = criterion(
    test_logits,
    test_labels
)


print("=" * 100)
print("LOSS VERIFICATION")
print("=" * 100)

print("Logits shape :", tuple(test_logits.shape))
print("Labels shape :", tuple(test_labels.shape))

print("Test loss    :", test_loss.item())

print(
    "Loss finite  :",
    bool(torch.isfinite(test_loss))
)

LOSS VERIFICATION
Logits shape : (32,)
Labels shape : (32,)
Test loss    : 0.6417804956436157
Loss finite  : True


In [9]:
# ============================================================
# CELL 9 — OPTIMIZER CONFIGURATION
# ============================================================

LEARNING_RATE = 1e-3

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


print("=" * 100)
print("OPTIMIZER CONFIGURATION")
print("=" * 100)

print("Optimizer     : Adam")
print("Learning rate :", LEARNING_RATE)

OPTIMIZER CONFIGURATION
Optimizer     : Adam
Learning rate : 0.001


In [10]:
# ============================================================
# CELL 10 — DATASETS AND DETERMINISTIC DATALOADERS
# ============================================================

class CrossModalPairDataset(Dataset):

    def __init__(
        self,
        source_embeddings,
        target_embeddings,
        labels
    ):

        self.source = torch.tensor(
            source_embeddings,
            dtype=torch.float32
        )

        self.target = torch.tensor(
            target_embeddings,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):

        return (
            self.source[index],
            self.target[index],
            self.labels[index]
        )


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = CrossModalPairDataset(
    train_source,
    train_target,
    train_labels
)

val_dataset = CrossModalPairDataset(
    val_source,
    val_target,
    val_labels
)


# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

BATCH_SIZE = 32


# ------------------------------------------------------------
# Dedicated deterministic generator
# ------------------------------------------------------------

train_generator = torch.Generator()

train_generator.manual_seed(SEED)


# ------------------------------------------------------------
# Create DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print("=" * 100)
print("DATALOADERS CREATED")
print("=" * 100)

print("Training samples   :", len(train_dataset))
print("Validation samples :", len(val_dataset))

print("Training batches   :", len(train_loader))
print("Validation batches :", len(val_loader))

print("Batch size         :", BATCH_SIZE)

print("\nTraining shuffle   : True")
print("Validation shuffle : False")
print("DataLoader seed    :", SEED)

DATALOADERS CREATED
Training samples   : 1776
Validation samples : 448
Training batches   : 56
Validation batches : 14
Batch size         : 32

Training shuffle   : True
Validation shuffle : False
DataLoader seed    : 42


In [11]:
# ============================================================
# CELL 11 — SINGLE-STEP TRAINING SANITY CHECK
# ============================================================

model.train()


# ------------------------------------------------------------
# Get first deterministic training batch
# ------------------------------------------------------------

source_batch, target_batch, label_batch = next(
    iter(train_loader)
)

source_batch = source_batch.to(device)
target_batch = target_batch.to(device)
label_batch = label_batch.to(device)


print("=" * 100)
print("SINGLE-STEP TRAINING SANITY CHECK")
print("=" * 100)

print("Source batch shape :", tuple(source_batch.shape))
print("Target batch shape :", tuple(target_batch.shape))
print("Label batch shape  :", tuple(label_batch.shape))

print(
    "\nPositive labels:",
    int((label_batch == 1).sum().item())
)

print(
    "Negative labels:",
    int((label_batch == 0).sum().item())
)


# ------------------------------------------------------------
# Save model parameter before update
# ------------------------------------------------------------

parameter_before = (
    model.classifier[0].weight
    .detach()
    .clone()
)


# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

optimizer.zero_grad()

batch_logits = model(
    source_batch,
    target_batch
)


# ------------------------------------------------------------
# Calculate loss
# ------------------------------------------------------------

batch_loss = criterion(
    batch_logits,
    label_batch
)


# ------------------------------------------------------------
# Backpropagation
# ------------------------------------------------------------

batch_loss.backward()


# ------------------------------------------------------------
# Check gradients
# ------------------------------------------------------------

gradient_count = 0
all_gradients_finite = True

for parameter in model.parameters():

    if parameter.grad is not None:

        gradient_count += 1

        if not torch.all(
            torch.isfinite(parameter.grad)
        ):

            all_gradients_finite = False


# ------------------------------------------------------------
# Perform optimizer update
# ------------------------------------------------------------

optimizer.step()


# ------------------------------------------------------------
# Measure parameter change
# ------------------------------------------------------------

parameter_after = (
    model.classifier[0].weight
    .detach()
    .clone()
)

parameter_change = torch.abs(
    parameter_after - parameter_before
).sum().item()


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\n" + "-" * 100)
print("LEARNING STEP RESULTS")
print("-" * 100)

print("Logits shape       :", tuple(batch_logits.shape))
print("Batch loss         :", batch_loss.item())
print("Loss finite        :", bool(torch.isfinite(batch_loss)))

print("Parameters with gradients:",
      gradient_count)

print("All gradients finite:",
      all_gradients_finite)

print("Parameter change   :",
      parameter_change)

print("Weights changed    :",
      parameter_change > 0)


# ------------------------------------------------------------
# Final sanity assertions
# ------------------------------------------------------------

assert batch_logits.shape == (32,)
assert torch.isfinite(batch_loss)
assert gradient_count == 6
assert all_gradients_finite
assert parameter_change > 0


print("\n" + "=" * 100)
print("✓ SINGLE-STEP TRAINING SANITY CHECK PASSED")
print("=" * 100)

SINGLE-STEP TRAINING SANITY CHECK
Source batch shape : (32, 32)
Target batch shape : (32, 32)
Label batch shape  : (32,)

Positive labels: 21
Negative labels: 11

----------------------------------------------------------------------------------------------------
LEARNING STEP RESULTS
----------------------------------------------------------------------------------------------------
Logits shape       : (32,)
Batch loss         : 0.6528235673904419
Loss finite        : True
Parameters with gradients: 6
All gradients finite: True
Parameter change   : 14.047223091125488
Weights changed    : True

✓ SINGLE-STEP TRAINING SANITY CHECK PASSED


In [12]:
# ============================================================
# CELL 12 — CLEAN RESET FOR OFFICIAL TRAINING
# ============================================================

import os
import torch


# ------------------------------------------------------------
# 1. Recreate a completely fresh model
# ------------------------------------------------------------

model = BaselineCrossModalModel().to(device)


# ------------------------------------------------------------
# 2. Recreate optimizer
# ------------------------------------------------------------

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


# ------------------------------------------------------------
# 3. Reset deterministic DataLoader generator
# ------------------------------------------------------------

train_generator = torch.Generator()
train_generator.manual_seed(SEED)


# ------------------------------------------------------------
# 4. Recreate training DataLoader
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator
)


# ------------------------------------------------------------
# 5. Recreate validation DataLoader
# ------------------------------------------------------------

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ------------------------------------------------------------
# 6. Create checkpoint directory
# ------------------------------------------------------------

checkpoint_dir = os.path.join(
    project_root,
    "06_cross_modal_learning",
    "03_cross_modal_learning",
    "03_04_baseline_training_reproducible",
    "checkpoints"
)

os.makedirs(
    checkpoint_dir,
    exist_ok=True
)


# ------------------------------------------------------------
# 7. Verify fresh model parameters
# ------------------------------------------------------------

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)


trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print("=" * 100)
print("CLEAN TRAINING RESET")
print("=" * 100)

print("Model recreated       : True")
print("Optimizer recreated   : True")
print("DataLoader seed       :", SEED)

print("\nTotal parameters      :", total_parameters)
print("Trainable parameters  :", trainable_parameters)

print("\nCheckpoint directory:")
print(checkpoint_dir)

print("\nCheckpoint directory exists:",
      os.path.exists(checkpoint_dir))


# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------

assert isinstance(model, BaselineCrossModalModel)
assert isinstance(optimizer, torch.optim.Adam)

assert os.path.isdir(checkpoint_dir)

assert total_parameters == trainable_parameters

print("\n" + "=" * 100)
print("✓ CLEAN RESET PASSED — READY FOR OFFICIAL TRAINING")
print("=" * 100)

CLEAN TRAINING RESET
Model recreated       : True
Optimizer recreated   : True
DataLoader seed       : 42

Total parameters      : 24833
Trainable parameters  : 24833

Checkpoint directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_04_baseline_training_reproducible\checkpoints

Checkpoint directory exists: True

✓ CLEAN RESET PASSED — READY FOR OFFICIAL TRAINING


In [13]:
# ============================================================
# CELL 13 — OFFICIAL TRAINING CONFIGURATION
# ============================================================

NUM_EPOCHS = 50

BEST_MODEL_PATH = os.path.join(
    checkpoint_dir,
    "03_04_baseline_best_model.pt"
)

FINAL_MODEL_PATH = os.path.join(
    checkpoint_dir,
    "03_04_baseline_final_model.pt"
)

HISTORY_PATH = os.path.join(
    checkpoint_dir,
    "03_04_baseline_training_history.pt"
)


# ------------------------------------------------------------
# Metric tracking containers
# ------------------------------------------------------------

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_balanced_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
    "val_roc_auc": []
}


# ------------------------------------------------------------
# Best-model tracking
# ------------------------------------------------------------

best_val_roc_auc = -float("inf")
best_epoch = None


# ------------------------------------------------------------
# Training configuration summary
# ------------------------------------------------------------

print("=" * 100)
print("OFFICIAL BASELINE TRAINING CONFIGURATION")
print("=" * 100)

print("Number of epochs      :", NUM_EPOCHS)
print("Batch size            :", BATCH_SIZE)
print("Learning rate         :", LEARNING_RATE)
print("Optimizer             : Adam")
print("Loss function         : BCEWithLogitsLoss")
print("Random seed           :", SEED)

print("\nBest model path:")
print(BEST_MODEL_PATH)

print("\nFinal model path:")
print(FINAL_MODEL_PATH)

print("\nHistory path:")
print(HISTORY_PATH)

print("\n" + "=" * 100)
print("✓ TRAINING CONFIGURATION READY")
print("=" * 100)

OFFICIAL BASELINE TRAINING CONFIGURATION
Number of epochs      : 50
Batch size            : 32
Learning rate         : 0.001
Optimizer             : Adam
Loss function         : BCEWithLogitsLoss
Random seed           : 42

Best model path:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_04_baseline_training_reproducible\checkpoints\03_04_baseline_best_model.pt

Final model path:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_04_baseline_training_reproducible\checkpoints\03_04_baseline_final_model.pt

History path:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_04_baseline_training_reproducible\checkpoints\03_04_baseline_training_history.pt

✓ TRAINING CONFIGURATION READY


In [14]:
# ============================================================
# CELL 14 — OFFICIAL BASELINE TRAINING
# ============================================================

import copy
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

print("=" * 100)
print("OFFICIAL BASELINE TRAINING")
print("=" * 100)

for epoch in range(1, NUM_EPOCHS + 1):

    # ========================================================
    # TRAINING
    # ========================================================

    model.train()

    running_train_loss = 0.0
    train_sample_count = 0

    for source_batch, target_batch, label_batch in train_loader:

        source_batch = source_batch.to(device)
        target_batch = target_batch.to(device)
        label_batch = label_batch.to(device)

        optimizer.zero_grad()

        logits = model(
            source_batch,
            target_batch
        )

        loss = criterion(
            logits,
            label_batch
        )

        loss.backward()

        optimizer.step()

        batch_size = label_batch.size(0)

        running_train_loss += (
            loss.item() * batch_size
        )

        train_sample_count += batch_size

    train_loss = (
        running_train_loss /
        train_sample_count
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    running_val_loss = 0.0
    val_sample_count = 0

    all_val_labels = []
    all_val_probabilities = []

    with torch.no_grad():

        for source_batch, target_batch, label_batch in val_loader:

            source_batch = source_batch.to(device)
            target_batch = target_batch.to(device)
            label_batch = label_batch.to(device)

            logits = model(
                source_batch,
                target_batch
            )

            loss = criterion(
                logits,
                label_batch
            )

            probabilities = torch.sigmoid(logits)

            batch_size = label_batch.size(0)

            running_val_loss += (
                loss.item() * batch_size
            )

            val_sample_count += batch_size

            all_val_labels.extend(
                label_batch.cpu().numpy()
            )

            all_val_probabilities.extend(
                probabilities.cpu().numpy()
            )


    val_loss = (
        running_val_loss /
        val_sample_count
    )


    # ========================================================
    # CONVERT VALIDATION OUTPUTS
    # ========================================================

    all_val_labels = np.asarray(
        all_val_labels
    )

    all_val_probabilities = np.asarray(
        all_val_probabilities
    )

    all_val_predictions = (
        all_val_probabilities >= 0.5
    ).astype(int)


    # ========================================================
    # VALIDATION METRICS
    # ========================================================

    val_accuracy = accuracy_score(
        all_val_labels,
        all_val_predictions
    )

    val_balanced_accuracy = balanced_accuracy_score(
        all_val_labels,
        all_val_predictions
    )

    val_precision = precision_score(
        all_val_labels,
        all_val_predictions,
        zero_division=0
    )

    val_recall = recall_score(
        all_val_labels,
        all_val_predictions,
        zero_division=0
    )

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        zero_division=0
    )

    val_roc_auc = roc_auc_score(
        all_val_labels,
        all_val_probabilities
    )


    # ========================================================
    # STORE HISTORY
    # ========================================================

    history["epoch"].append(epoch)

    history["train_loss"].append(
        float(train_loss)
    )

    history["val_loss"].append(
        float(val_loss)
    )

    history["val_accuracy"].append(
        float(val_accuracy)
    )

    history["val_balanced_accuracy"].append(
        float(val_balanced_accuracy)
    )

    history["val_precision"].append(
        float(val_precision)
    )

    history["val_recall"].append(
        float(val_recall)
    )

    history["val_f1"].append(
        float(val_f1)
    )

    history["val_roc_auc"].append(
        float(val_roc_auc)
    )


    # ========================================================
    # SAVE BEST MODEL IMMEDIATELY
    # ========================================================

    is_new_best = (
        val_roc_auc > best_val_roc_auc
    )

    if is_new_best:

        best_val_roc_auc = float(
            val_roc_auc
        )

        best_epoch = epoch

        best_checkpoint = {
            "epoch": epoch,
            "model_state_dict": copy.deepcopy(
                model.state_dict()
            ),
            "optimizer_state_dict": copy.deepcopy(
                optimizer.state_dict()
            ),
            "best_val_roc_auc": best_val_roc_auc,
            "seed": SEED,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "num_epochs": NUM_EPOCHS
        }

        torch.save(
            best_checkpoint,
            BEST_MODEL_PATH
        )


    # ========================================================
    # PRINT EPOCH RESULTS
    # ========================================================

    marker = "  <-- BEST" if is_new_best else ""

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Acc: {val_accuracy:.4f} | "
        f"F1: {val_f1:.4f} | "
        f"ROC-AUC: {val_roc_auc:.4f}"
        f"{marker}"
    )


# ============================================================
# SAVE FINAL MODEL
# ============================================================

final_checkpoint = {
    "epoch": NUM_EPOCHS,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS
}

torch.save(
    final_checkpoint,
    FINAL_MODEL_PATH
)


# ============================================================
# SAVE TRAINING HISTORY
# ============================================================

torch.save(
    history,
    HISTORY_PATH
)


# ============================================================
# FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("OFFICIAL TRAINING COMPLETE")
print("=" * 100)

print("Total epochs trained :", NUM_EPOCHS)
print("Best epoch           :", best_epoch)
print("Best validation ROC-AUC:",
      f"{best_val_roc_auc:.6f}")

print("\nBest checkpoint saved:")
print(BEST_MODEL_PATH)

print("\nFinal checkpoint saved:")
print(FINAL_MODEL_PATH)

print("\nTraining history saved:")
print(HISTORY_PATH)

print("\n" + "=" * 100)
print("✓ OFFICIAL BASELINE TRAINING COMPLETED")
print("=" * 100)

OFFICIAL BASELINE TRAINING
Epoch 01/50 | Train Loss: 0.6970 | Val Loss: 0.6924 | Acc: 0.5112 | F1: 0.3178 | ROC-AUC: 0.5267  <-- BEST
Epoch 02/50 | Train Loss: 0.6923 | Val Loss: 0.6926 | Acc: 0.5246 | F1: 0.6718 | ROC-AUC: 0.5442  <-- BEST
Epoch 03/50 | Train Loss: 0.6887 | Val Loss: 0.6865 | Acc: 0.5603 | F1: 0.4775 | ROC-AUC: 0.5834  <-- BEST
Epoch 04/50 | Train Loss: 0.6801 | Val Loss: 0.6810 | Acc: 0.5603 | F1: 0.4632 | ROC-AUC: 0.5898  <-- BEST
Epoch 05/50 | Train Loss: 0.6686 | Val Loss: 0.6884 | Acc: 0.5357 | F1: 0.1938 | ROC-AUC: 0.6078  <-- BEST
Epoch 06/50 | Train Loss: 0.6550 | Val Loss: 0.6697 | Acc: 0.5871 | F1: 0.6463 | ROC-AUC: 0.6223  <-- BEST
Epoch 07/50 | Train Loss: 0.6427 | Val Loss: 0.6585 | Acc: 0.6094 | F1: 0.5995 | ROC-AUC: 0.6468  <-- BEST
Epoch 08/50 | Train Loss: 0.6467 | Val Loss: 0.6705 | Acc: 0.5938 | F1: 0.4678 | ROC-AUC: 0.6467
Epoch 09/50 | Train Loss: 0.6346 | Val Loss: 0.6470 | Acc: 0.6228 | F1: 0.6302 | ROC-AUC: 0.6609  <-- BEST
Epoch 10/50 | Train 

In [15]:
# ============================================================
# CELL 15 — BEST CHECKPOINT VERIFICATION
# ============================================================

import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Load best checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False
)


# ------------------------------------------------------------
# 2. Recreate model architecture
# ------------------------------------------------------------

best_model = BaselineCrossModalModel().to(device)

best_model.load_state_dict(
    checkpoint["model_state_dict"]
)

best_model.eval()


# ------------------------------------------------------------
# 3. Validate saved checkpoint
# ------------------------------------------------------------

all_labels = []
all_probabilities = []

running_loss = 0.0
sample_count = 0


with torch.no_grad():

    for source_batch, target_batch, label_batch in val_loader:

        source_batch = source_batch.to(device)
        target_batch = target_batch.to(device)
        label_batch = label_batch.to(device)

        logits = best_model(
            source_batch,
            target_batch
        )

        loss = criterion(
            logits,
            label_batch
        )

        probabilities = torch.sigmoid(logits)

        batch_size = label_batch.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        sample_count += batch_size

        all_labels.extend(
            label_batch.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )


# ------------------------------------------------------------
# 4. Convert outputs
# ------------------------------------------------------------

all_labels = np.asarray(
    all_labels
)

all_probabilities = np.asarray(
    all_probabilities
)

all_predictions = (
    all_probabilities >= 0.5
).astype(int)


# ------------------------------------------------------------
# 5. Calculate metrics
# ------------------------------------------------------------

verified_val_loss = (
    running_loss /
    sample_count
)

verified_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

verified_balanced_accuracy = balanced_accuracy_score(
    all_labels,
    all_predictions
)

verified_precision = precision_score(
    all_labels,
    all_predictions,
    zero_division=0
)

verified_recall = recall_score(
    all_labels,
    all_predictions,
    zero_division=0
)

verified_f1 = f1_score(
    all_labels,
    all_predictions,
    zero_division=0
)

verified_roc_auc = roc_auc_score(
    all_labels,
    all_probabilities
)

verified_cm = confusion_matrix(
    all_labels,
    all_predictions
)


# ------------------------------------------------------------
# 6. Compare checkpoint metadata with evaluation
# ------------------------------------------------------------

saved_best_epoch = checkpoint["epoch"]
saved_best_roc_auc = checkpoint["best_val_roc_auc"]


print("=" * 100)
print("BEST CHECKPOINT VERIFICATION")
print("=" * 100)

print("Saved checkpoint epoch :", saved_best_epoch)

print(
    "Saved ROC-AUC          :",
    f"{saved_best_roc_auc:.6f}"
)

print(
    "Verified ROC-AUC       :",
    f"{verified_roc_auc:.6f}"
)

print(
    "Verified Val Loss      :",
    f"{verified_val_loss:.6f}"
)

print(
    "Verified Accuracy      :",
    f"{verified_accuracy:.6f}"
)

print(
    "Verified Balanced Acc  :",
    f"{verified_balanced_accuracy:.6f}"
)

print(
    "Verified Precision     :",
    f"{verified_precision:.6f}"
)

print(
    "Verified Recall        :",
    f"{verified_recall:.6f}"
)

print(
    "Verified F1            :",
    f"{verified_f1:.6f}"
)

print("\nConfusion Matrix:")
print(verified_cm)


# ------------------------------------------------------------
# 7. Check consistency
# ------------------------------------------------------------

roc_auc_difference = abs(
    saved_best_roc_auc -
    verified_roc_auc
)

print("\nROC-AUC difference:",
      f"{roc_auc_difference:.12f}")


assert saved_best_epoch == best_epoch

assert roc_auc_difference < 1e-6

assert len(all_labels) == 448

assert all_probabilities.shape == (448,)

assert np.all(
    np.isfinite(all_probabilities)
)


print("\n" + "=" * 100)
print("✓ BEST CHECKPOINT VERIFIED SUCCESSFULLY")
print("=" * 100)

BEST CHECKPOINT VERIFICATION
Saved checkpoint epoch : 13
Saved ROC-AUC          : 0.662667
Verified ROC-AUC       : 0.662667
Verified Val Loss      : 0.660638
Verified Accuracy      : 0.611607
Verified Balanced Acc  : 0.611607
Verified Precision     : 0.606838
Verified Recall        : 0.633929
Verified F1            : 0.620087

Confusion Matrix:
[[132  92]
 [ 82 142]]

ROC-AUC difference: 0.000000000000

✓ BEST CHECKPOINT VERIFIED SUCCESSFULLY
